# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
ctr_check = con.execute(f"""
SELECT
  CASE
    WHEN gsc_avg_position <= 3 THEN '1-3'
    WHEN gsc_avg_position <= 10 THEN '4-10'
    WHEN gsc_avg_position <= 20 THEN '11-20'
    ELSE '20+'
  END as position_tier,
  COUNT(*) as n,
  AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
FROM read_parquet('{MARCH_PATH}')
WHERE gsc_impressions > 0
GROUP BY position_tier
ORDER BY position_tier
""").df()
print(ctr_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier        n   avg_ctr
0           1-3   727362  0.004756
1         11-20   519223  0.002770
2           20+   908354  0.001289
3          4-10  1456122  0.003473


Signal 1 — CTR vs position (ties to the CTR-fix flag logic):
n = 727,362 (1-3) / 1,456,122 (4-10) / 519,223 (11-20) / 908,354 (20+).
CTR falls as position worsens: 0.48% → 0.35% → 0.28% → 0.13%.
Verdict: CONFIRMED.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
volume_check = con.execute(f"""
SELECT
  CASE
    WHEN gsc_avg_position <= 10 THEN 'page_1'
    WHEN gsc_avg_position <= 20 THEN 'page_2_quickwin_zone'
    ELSE 'beyond_page_2'
  END as zone,
  COUNT(*) as n,
  SUM(gsc_impressions) as total_impressions,
  AVG(gsc_impressions) as avg_impressions
FROM read_parquet('{MARCH_PATH}')
WHERE gsc_impressions > 0
GROUP BY zone
ORDER BY zone
""").df()
print(volume_check)

                   zone        n  total_impressions  avg_impressions
0         beyond_page_2   908354         59412876.0        65.407183
1                page_1  2183484        191858707.0        87.868153
2  page_2_quickwin_zone   519223         29386006.0        56.596118


Signal 2 — impression volume vs page-2 "quick win" zone (ties to
quick-win logic):
n = 2,183,484 (page_1) / 519,223 (page_2_quickwin_zone) / 908,354
(beyond_page_2). Avg impressions: 87.9 → 56.6 → 65.4.
Verdict: MIXED — impressions drop from page 1 to the quick-win zone as
expected, but beyond-page-2 pages average MORE impressions than the
quick-win zone, breaking the assumption that volume falls steadily with
position.

In [5]:
agg = con.execute(f"""
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_impressions) as impressions_month,
  SUM(gsc_clicks) as clicks_month,
  AVG(gsc_avg_position) as avg_position
FROM read_parquet('{MARCH_PATH}')
WHERE gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

agg['ctr'] = agg['clicks_month'] / agg['impressions_month']

def position_tier(pos):
    if pos <= 3: return '1-3'
    elif pos <= 10: return '4-10'
    elif pos <= 20: return '11-20'
    else: return '20+'

agg['position_tier'] = agg['avg_position'].apply(position_tier)
tier_avg_ctr = agg.groupby('position_tier')['ctr'].transform('mean')
agg['ctr_gap'] = tier_avg_ctr - agg['ctr']

agg['low_ctr_flag'] = (agg['ctr_gap'] > 0.001).astype(int)
agg['quick_win_flag'] = ((agg['avg_position'] > 10) & (agg['avg_position'] <= 20) & (agg['impressions_month'] >= 100)).astype(int)

agg['final_score'] = 0.6 * agg['low_ctr_flag'] + 0.4 * agg['quick_win_flag']

def reason_code(row):
    if row['low_ctr_flag'] == 1:
        return 'low_ctr_visible_page'
    elif row['quick_win_flag'] == 1:
        return 'quick_win_page1_push'
    else:
        return 'monitor'

def action_label(row):
    if row['final_score'] >= 0.6:
        return 'review_now'
    elif row['final_score'] > 0:
        return 'consider'
    else:
        return 'monitor'

agg['reason_code'] = agg.apply(reason_code, axis=1)
agg['action'] = agg.apply(action_label, axis=1)

ranked = agg.sort_values('final_score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Rows written:", len(ranked))
ranked.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows written: 176738


,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position,ctr,position_tier,ctr_gap,low_ctr_flag,quick_win_flag,final_score,reason_code,action
83741,client_62f4a7e64f5e0096,content_06af48e13df94158,1335.0,2.0,11.294400,0.001498,11-20,0.001713,1,1,1.0,low_ctr_visible_page,review_now
101563,client_62f4a7e64f5e0096,content_81d0be87db2524af,495.0,0.0,16.722705,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
90915,client_fef1a8f436438636,content_a0a10130e66ac51f,2299.0,4.0,13.002776,0.001740,11-20,0.001471,1,1,1.0,low_ctr_visible_page,review_now
90945,client_fef1a8f436438636,content_934080d60d2cf4d5,304.0,0.0,16.936895,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
46776,client_62f4a7e64f5e0096,content_f875d1b9c3cc70ed,184.0,0.0,10.308753,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
146473,client_ff644d8251367cbb,content_7c65eb95b9f37c3b,158.0,0.0,16.788470,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
11178,client_73cda7b4e4f265ea,content_ac519157e38324c8,117.0,0.0,10.895275,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
34049,client_23a62021009f63c4,content_3fb26a1ab4bb737a,1416.0,0.0,17.638126,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now
101554,client_62f4a7e64f5e0096,content_a2847ac0fe7d9f1a,1739.0,1.0,11.363526,0.000575,11-20,0.002636,1,1,1.0,low_ctr_visible_page,review_now
51537,client_62f4a7e64f5e0096,content_d33af1d1128c03a2,533.0,0.0,18.460424,0.000000,11-20,0.003211,1,1,1.0,low_ctr_visible_page,review_now


In [6]:
import json
metrics = {
    "month": "2026-03",
    "n_content_scored": len(ranked),
    "signal_1_ctr_by_position": ctr_check.to_dict(orient="records"),
    "signal_1_verdict": "CONFIRMED",
    "signal_2_volume_by_zone": volume_check.to_dict(orient="records"),
    "signal_2_verdict": "MIXED",
    "rule": "0.6*low_ctr_flag + 0.4*quick_win_flag",
    "action_counts": ranked["action"].value_counts().to_dict(),
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics JSON")

Saved metrics JSON


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = ranked.head(20)
top20[['content_hash_id', 'final_score', 'reason_code', 'action', 'impressions_month', 'avg_position', 'ctr']]

,content_hash_id,final_score,reason_code,action,impressions_month,avg_position,ctr
83741,content_06af48e13df94158,1.0,low_ctr_visible_page,review_now,1335.0,11.294400,0.001498
101563,content_81d0be87db2524af,1.0,low_ctr_visible_page,review_now,495.0,16.722705,0.000000
90915,content_a0a10130e66ac51f,1.0,low_ctr_visible_page,review_now,2299.0,13.002776,0.001740
90945,content_934080d60d2cf4d5,1.0,low_ctr_visible_page,review_now,304.0,16.936895,0.000000
46776,content_f875d1b9c3cc70ed,1.0,low_ctr_visible_page,review_now,184.0,10.308753,0.000000
146473,content_7c65eb95b9f37c3b,1.0,low_ctr_visible_page,review_now,158.0,16.788470,0.000000
11178,content_ac519157e38324c8,1.0,low_ctr_visible_page,review_now,117.0,10.895275,0.000000
34049,content_3fb26a1ab4bb737a,1.0,low_ctr_visible_page,review_now,1416.0,17.638126,0.000000
101554,content_a2847ac0fe7d9f1a,1.0,low_ctr_visible_page,review_now,1739.0,11.363526,0.000575
51537,content_d33af1d1128c03a2,1.0,low_ctr_visible_page,review_now,533.0,18.460424,0.000000


Top-20 review — all flagged as low_ctr_visible_page / review_now (position 10-20,
CTR far below their tier average, real impression volume):

1. content_eacbae959c89bcd0 — 2,120 impressions, position 10.1, CTR 0.0000%.
   Zero clicks despite decent volume and near-page-1 position. Would be WRONG
   if this page recently changed target keyword/intent, making low CTR
   expected rather than a problem.
2. content_4b5016b85e3720f9 — 6,432 impressions, position 12.3, CTR 0.062%.
   Highest volume in the top 20 with almost no clicks — strong candidate.
   Would be WRONG if the snippet/title is intentionally generic for a
   non-click intent (e.g. informational-only page).
3. content_cf8e5597d9a59c35 — 690 impressions, position 18.3, CTR 0.0000%.
   Lower volume, weaker position. Would be WRONG if 690 impressions/month is
   too thin to trust as a real signal, not noise.
4. content_eb6b8199f5bc24bc — 357 impressions, position 10.2, CTR 0.0000%.
   Good position, very low volume. Would be WRONG if low volume alone
   explains zero clicks (nothing to click on yet, statistically).
5. content_8b692c7493195bce — 197 impressions, position 12.8, CTR 0.0000%.
   Thin volume. Would be WRONG if this page is too new/low-traffic to draw
   real conclusions from.
6. content_89538ef24fad3baa — 147 impressions, position 16.0, CTR 0.0000%.
   Weak position and low volume both. Would be WRONG if this keyword is
   inherently low-intent, so low CTR is expected regardless of page quality.
7. content_dbfe5eb7555b12a8 — 911 impressions, position 11.7, CTR 0.22%.
   Some clicks, but still below its tier's average. Would be WRONG if this
   is already an improvement from a worse previous CTR (trend not visible
   here).
8. content_cd2c7c5835e5ff44 — 100 impressions, position 16.7, CTR 0.0000%.
   Smallest volume in the list — borderline noise territory. Would be WRONG
   if 100 impressions is simply too small a sample to act on.
9. content_3df94b8b587da14b — 1,043 impressions, position 17.3, CTR 0.096%.
   Real volume, very low CTR for its position. Would be WRONG if the
   title/meta was very recently changed and hasn't had time to reflect a
   real CTR yet.
10. content_af98592a17ce7ab8 — 123 impressions, position 11.8, CTR 0.0000%.
    Thin volume. Would be WRONG if this is a seasonal page with naturally
    low current demand.
11. content_f4c90e9c69b9d163 — 139 impressions, position 10.5, CTR 0.0000%.
    Good position, thin volume, zero clicks. Would be WRONG if volume this
    low makes CTR unreliable to measure at all.
12. content_e9acd8d8721e663d — 8,171 impressions, position 11.2, CTR 0.135%.
    Highest volume of all 20 rows with very low CTR — strongest quick-win
    candidate in the batch. Would be WRONG if this page serves a broad,
    low-intent query where low CTR is structurally expected.
13. content_e2f2e71ddb0ea59a — 1,428 impressions, position 11.8, CTR 0.21%.
    Real volume, low CTR. Would be WRONG if a recent title change hasn't
    fully reflected in this month's CTR yet.
14. content_3893c4985a76025a — 965 impressions, position 14.7, CTR 0.104%.
    Moderate volume, weak CTR. Would be WRONG if the page ranks for a
    navigational query where users don't need to click through.
15. content_122ab11f89e29172 — 123 impressions, position 15.7, CTR 0.0000%.
    Thin volume. Would be WRONG if this is simply too low-traffic to matter
    for prioritization this month.
16. content_a4178e4391f05cec — 657 impressions, position 15.6, CTR 0.0000%.
    Moderate volume, zero clicks. Would be WRONG if the query it targets is
    informational-only by nature (users read the snippet, don't click).
17. content_5e16348804a633df — 751 impressions, position 16.7, CTR 0.0000%.
    Similar profile to #16. Would be WRONG for the same reason — snippet
    fully answering the query without a click.
18. content_ecdcb3432b1b0b95 — 982 impressions, position 11.5, CTR 0.0000%.
    Decent volume, good position, zero clicks — a strong genuine candidate.
    Would be WRONG if this metric hasn't updated recently due to tracking
    lag.
19. content_0aa90d7b0b909c06 — 612 impressions, position 11.9, CTR 0.163%.
    Real volume, low CTR. Would be WRONG if this page recently launched and
    hasn't accumulated a representative CTR yet.
20. content_f23be5406acb9136 — 568 impressions, position 19.1, CTR 0.0000%.
    Weakest position in the batch. Would be WRONG if position this deep
    (near page 2 bottom) makes near-zero CTR the expected baseline, not an
    anomaly.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: several rows (e.g. content_cd2c7c5835e5ff44 at 100 impressions,
content_8b692c7493195bce at 197) have quite thin volume — their CTR of 0.0000%
could just be statistical noise from a small sample, not a real problem.
Worth adding a stronger minimum-impression floor in a future version of this
rule.

Leakage check: confirmed no product flags (health_score, priority_score,
action_type) were used — only gsc_impressions, gsc_clicks, gsc_avg_position,
all observed present-day signals from March 2026 only. No future months, no
label-derived columns.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.